# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kishiagaytano/wilt/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Ranking / scoring**

The decision is "which pages does an editor open first," under a capacity limit of a few dozen per sprint. That is an ordering problem: I need the *top* of a list to be right, not every row labelled correctly.

Classification is the wrong fit for a specific reason. There is no observed outcome column in this data that says a page was reviewed, fixed, or should have been — the release deliberately ships observable signals only, and no product decision flags. To classify, I would first have to invent a binary label ("gap worse than X"), then train a model to reproduce it. That is a circular result: I would be measuring how well a model copies my own rule, not discovering anything. Clustering is out for the same practical reason — it produces groups, not an order, and I need an order.

So: score every eligible page by how far its click rate falls below what its search position would predict, then rank. The output is a queue, and the metric has to be a top-of-list metric.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL, REPO_DIR = "https://github.com/kishiagaytano/wilt", "wilt"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

TIER_ORDER = ["top_3", "page_1", "striking", "page_3_5", "deep"]
FLOOR = 1000                      # illustrative — the policy choice belongs to ML-04
sel = df[(df["impressions_90d"] >= FLOOR) & (df["avg_position"] > 0)].copy()

# Is there any observed outcome to classify against?
outcome_cols = [c for c in df.columns if any(k in c.lower() for k in
                ("action", "priority", "health", "review", "flag", "fixed", "resolved"))]
print("columns recording a human review decision or outcome:", outcome_cols or "none")
print(f"eligible pages to order : {len(sel):,}")
print(f"reviewer capacity ~50   : the top {50/len(sel):.2%} of the list is what actually gets used")

columns recording a human review decision or outcome: none
eligible pages to order : 13,512
reviewer capacity ~50   : the top 0.37% of the list is what actually gets used


## 2. Target or proxy

**What I score:** the expected number of clicks a page should earn given its impressions and its search position, and the shortfall between that and what it actually earned.

**Where it comes from — honestly, a derived score, not an observed label.** The *inputs* are observed measurements: `clicks_90d` and `impressions_90d` are counted, `avg_position` is measured. The *expectation* is estimated from those same observations — the impression-weighted click rate of every page in the same position stratum. So the shortfall is derived from observed data, but it is not an observed outcome. Nothing in this dataset records whether a page was reviewed, or whether reviewing it helped.

By the guide's field types, this is a **target/proxy field I define myself**, and I have to say so out loud rather than dress it up as ground truth.

**The observed outcome that would upgrade it** is forward CTR movement: did a page I flagged move toward its position's expected rate over a later window? That needs the warehouse's daily facts, and it is the validation step in ML-09 — not something the starter slice can answer, because every column here describes one trailing 90-day window.

**Leakage rules for this lane, which differ from the starter's.** The starter forbids `trend_pct` and `trend_direction` because its label is derived from them. Those are harmless here. My forbidden pair is `ctr` and `clicks_90d` — they are the components of my own target, so they cannot be inputs to the expectation model. `impressions_90d` and `avg_position` stay legal: they are exposure and context, not outcome.

In [2]:
tier = sel.groupby("position_tier")[["clicks_90d", "impressions_90d"]].sum()
tier["expected_ctr"] = 100 * tier["clicks_90d"] / tier["impressions_90d"]
print("expectation estimated from observed counts, per position stratum:")
print(tier.reindex(TIER_ORDER).round(3).to_string())

print("\ntarget components — cannot be model features :", ["ctr", "clicks_90d"])
print("exposure / context — legal features          :", ["impressions_90d", "avg_position", "position_tier"])
print("starter label trap — irrelevant to this lane :", ["trend_pct", "trend_direction"])

expectation estimated from observed counts, per position stratum:
               clicks_90d  impressions_90d  expected_ctr
position_tier                                           
top_3               34105          6960840         0.490
page_1             309554         88424737         0.350
striking            77109         21804995         0.354
page_3_5            52991         33944083         0.156
deep                  342           944341         0.036

target components — cannot be model features : ['ctr', 'clicks_90d']
exposure / context — legal features          : ['impressions_90d', 'avg_position', 'position_tier']
starter label trap — irrelevant to this lane : ['trend_pct', 'trend_direction']


## 3. Success metric

**Primary metric: Precision@50 against a forward-observed outcome.** Of the top 50 pages my queue flags, what share show their click rate moving toward their position's expected rate over the following 30 days, measured on the warehouse daily facts. K=50 because that is roughly a sprint of editor capacity — the metric should match how the list is used.

**What "good" means, and why that number.** It has to beat the transparent flat-rule baseline by a margin larger than the instability of the measurement itself. I have a real estimate of that instability rather than a guess: in notebook 02 I re-ran a top-50 comparison across five client-holdout splits, and the same method swung 0.44–0.68 depending only on which clients landed in the holdout — an 18–24 point spread, wider than the 9-point gap between the two methods being compared. A different task, so the number does not transfer exactly, but it sets the order of magnitude: a few points of improvement on one split is not a result. I want a margin that survives repeated grouped splits, reported as a range and not a single figure.

**Two supporting checks, because one number is not enough for a ranking:**

- *Rank stability:* how much the top 50 overlaps across client-holdout splits. An unstable queue is not actionable even if its average score looks fine.
- *A hand review of the top 20.* If I cannot look at the top of my own list and see why each page is there, the reason codes are not doing their job.

**What I will not use.** Accuracy or ROC-AUC over the whole population — they describe the whole ranking, and nobody reads the whole ranking. And I will not score against a threshold I invented ("gap worse than X"), because a model trained to reproduce my own cutoff tells me only that it can reproduce my cutoff.

**Starter-slice caveat:** the forward outcome does not exist in this file. On the starter I can report stability and the hand review; the forward Precision@50 arrives with the warehouse.

In [3]:
sel["expected_ctr"] = sel["position_tier"].map(tier["expected_ctr"])
sel["gap_pp"] = sel["expected_ctr"] - sel["ctr"]

naive_top = sel.nlargest(50, "gap_pp")
print(f"pages tied at the largest raw gap : {(sel['gap_pp'] == sel['gap_pp'].max()).sum():,}")
print(f"median impressions, all eligible  : {sel['impressions_90d'].median():,.0f}")
print(f"median impressions, naive top 50  : {naive_top['impressions_90d'].median():,.0f}")
print(f"median clicks,      naive top 50  : {naive_top['clicks_90d'].median():,.0f}")
print("\n-> the naive gap ranks zero-click pages first: the top 50 has a median of 0 clicks")
print("   and lower median impressions than the population, so the raw gap rewards thin")
print("   evidence. the score must be volume-aware, measured at a volume floor.")

pages tied at the largest raw gap : 29
median impressions, all eligible  : 4,304
median impressions, naive top 50  : 2,066
median clicks,      naive top 50  : 0

-> the naive gap ranks zero-click pages first: the top 50 has a median of 0 clicks
   and lower median impressions than the population, so the raw gap rewards thin
   evidence. the score must be volume-aware, measured at a volume floor.


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item (a page), summarised over one trailing 90-day window**, restricted to pages with enough impressions to measure and a real position reading.

Three things that definition has to nail down:

- **`avg_position > 0` is a filter, not a threshold.** In this data `0` means "no position data", not "ranked first" — 1,205 rows carry it. Leaving them in would silently treat missing measurements as the best possible rank.
- **The volume floor is a policy choice**, shown here at 1,000 impressions for illustration and decided properly in ML-04. It is the main lever on the false-positive risk I described in ML-02.
- **`client_id` is for grouping only** — never a feature. It defines the holdout groups, because pages from one client share templates and topics, and a model that memorises a client would score well without generalising.

The grain check below matters more than it looks: if `content_id` repeated, every per-page aggregate downstream would silently double-count.

In [4]:
print("one row = one pseudonymized content item, trailing 90-day window")
print(f"rows: {len(sel):,}   clients: {sel['client_id'].nunique()}   columns: {sel.shape[1]}")
print(f"grain check — duplicate content_id rows: {int(sel['content_id'].duplicated().sum())}")
print(f"excluded: avg_position == 0 (no position data) -> {(df['avg_position'] == 0).sum():,} rows")

sel[["content_id", "client_id", "content_type", "impressions_90d",
     "clicks_90d", "ctr", "avg_position", "position_tier"]].head()

one row = one pseudonymized content item, trailing 90-day window
rows: 13,512   clients: 28   columns: 46
grain check — duplicate content_id rows: 0
excluded: avg_position == 0 (no position data) -> 1,205 rows


,content_id,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,position_tier
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,0.76,10.6,striking
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,0.05,20.3,page_3_5
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,page_3_5
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,0.49,6.2,page_1
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,0.13,44.0,page_3_5


## 5. Why ML beats a fixed rule here

The fixed rule already exists — `scripts/02_baseline_score.py` flags `ctr < 0.5%` at position 1–20 — and it fails in three separate ways I can measure.

**1. It flags most of the inventory.** On the eligible population it fires on well over half the pages. A queue that contains most of the list does not order anything, and the editor is back to guessing.

**2. One flat threshold cannot fit a curve that spans an order of magnitude.** Expected CTR runs from roughly 0.49% in `top_3` down to 0.04% in `deep` — about 14× end to end. A single cutoff is simultaneously far too lenient at the top of the results and impossible to meet at the bottom, so it mislabels both ends. The cross-tab below counts exactly how many pages it flags that are *not* under-capturing, and how many genuinely below-expectation pages it never sees.

**3. Position alone does not resolve it either — so this is not just "use a better rule."** Position tier explains only about 7% of the variance in per-page CTR on this slice, where `avg_position` is a 90-day mean — daily position data from the warehouse may explain more, and re-deriving this is part of the capstone. Roughly 93% of the spread sits *within* tiers. So the ranking has to weigh several signals at once — position, exposure, content type, intent — against an expectation estimated from data, and it has to discount pages whose evidence is thin. That last part is the piece no if-statement does: a zero-click page with 1,000 impressions and a zero-click page with 50,000 impressions in the same position tier produce the same raw gap and must not receive the same score.

This is what "the pattern is real but too messy to write by hand" means concretely here. It is not that a rule scores badly. It is that the rule is answering a different question from the one the editor is asking.

In [5]:
flat_flag = (sel["avg_position"] <= 20) & (sel["ctr"] < 0.5)
below_exp = sel["ctr"] < sel["expected_ctr"]

print(f"flat rule flags                       : {flat_flag.sum():,} ({flat_flag.mean():.1%} of eligible)")
print(f"  flagged but AT/ABOVE tier expectation: {(flat_flag & ~below_exp).sum():,}   <- false alarms")
print(f"below expectation but NOT flagged      : {(~flat_flag & below_exp).sum():,}   <- missed")

exp_min, exp_max = tier["expected_ctr"].min(), tier["expected_ctr"].max()
print(f"\ntier expectation spans {exp_max/exp_min:.0f}x ({exp_min:.3f}% to {exp_max:.3f}%)")
print("-> one flat CTR threshold cannot fit that range")

within = ((sel["ctr"] - sel.groupby("position_tier")["ctr"].transform("mean")) ** 2).sum()
total = ((sel["ctr"] - sel["ctr"].mean()) ** 2).sum()
print(f"position tier explains {1 - within/total:.1%} of CTR variance; "
      f"{within/total:.1%} is within-tier")

flat rule flags                       : 8,054 (59.6% of eligible)
  flagged but AT/ABOVE tier expectation: 1,139   <- false alarms
below expectation but NOT flagged      : 2,319   <- missed

tier expectation spans 14x (0.036% to 0.490%)
-> one flat CTR threshold cannot fit that range
position tier explains 6.8% of CTR variance; 93.2% is within-tier


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.